# Module 14 — Notebook 4: Mini Project — Refactor Messy Code

## Learning Objectives

By the end of this notebook, you will be able to:

- Identify research-coding problems in a real script (magic numbers, bad names, missing seed, no docstrings)
- Rewrite a messy analysis script using all of Module 14's habits
- Verify that the cleaned script is reproducible by building a checklist report

## Why This Matters for AI Research Engineering

Real-world research code often starts messy — written quickly during exploration, with shortcuts that made sense at the time. The skill of systematically improving code *without breaking it* is called **refactoring**, and it is a core engineering competency at safety labs.

This mini project puts together everything from Module 14: notebook-vs-script judgement, good naming, docstrings, named constants, seeds, and a reproducibility report.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_keys
Path('output').mkdir(exist_ok=True)
print("Setup complete.")

## The Messy Code

Below is a real-ish analysis script. It works, but it has several problems that make it hard to trust and re-run. Your job is to identify the problems and then rewrite the script cleanly.

In [ ]:
MESSY_CODE = """
import json
import random

with open('../../data/synthetic/model_outputs.json') as f:
    d = json.load(f)

x = [r for r in d if r['flagged'] == True]
y = len(x) / len(d)
if y > 0.3:
    print("BAD")
z = [r for r in d if len(r['response']) > 100]
print(len(z))
s = random.sample(d, 5)
for item in s:
    print(item['model'], item['response'][:50])
"""
print("Messy code loaded.")

## Step 1 — Identify the Problems

Look at `MESSY_CODE` carefully. There are four categories of problems:

- **`magic_numbers`** — literal values like `0.3`, `100`, `5` with no explanation
- **`bad_names`** — single-letter variable names (`d`, `x`, `y`, `z`, `s`) that hide intent
- **`no_docstring`** — there are no functions at all, so no docstrings
- **`no_seed`** — `random.sample()` is called without setting a seed first

Build an `issues` list that names all four problems.

In [ ]:
issues = [
    # List the four issue strings here
]

In [ ]:
check_type(issues, list, "issues is a list")
check_contains(issues, 'magic_numbers', "issues includes 'magic_numbers'")
check_contains(issues, 'bad_names', "issues includes 'bad_names'")
check_contains(issues, 'no_docstring', "issues includes 'no_docstring'")
check_contains(issues, 'no_seed', "issues includes 'no_seed'")

## Step 2 — Write the Clean Version

Rewrite the analysis as `clean_analysis.py` using all the Module 14 habits:

- Module-level docstring
- Named constants (`SEED`, `LONG_RESPONSE_THRESHOLD`, `SAMPLE_SIZE`, `FLAG_RATE_THRESHOLD`)
- Descriptive variable names
- Functions with docstrings and type hints
- `random.seed(SEED)` called before any random operation
- A `if __name__ == '__main__':` guard so the script can also be imported

The `%%writefile` magic will create the file for you — just fill in the code.

In [ ]:
%%writefile clean_analysis.py
"""clean_analysis.py — Reproducible model output analysis."""
import json
import random
from pathlib import Path

SEED = 42
LONG_RESPONSE_THRESHOLD = 100
SAMPLE_SIZE = 5
FLAG_RATE_THRESHOLD = 0.3

random.seed(SEED)


def load_outputs(path: str) -> list:
    """Load model outputs from a JSON file."""
    with open(path) as f:
        return json.load(f)


def compute_flag_rate(outputs: list) -> float:
    """Return the fraction of outputs that are flagged."""
    flagged = [r for r in outputs if r['flagged']]
    return len(flagged) / len(outputs)


def get_long_responses(outputs: list, threshold: int = LONG_RESPONSE_THRESHOLD) -> list:
    """Return outputs whose response text exceeds threshold characters."""
    return [r for r in outputs if len(r['response']) > threshold]


if __name__ == '__main__':
    outputs = load_outputs('../../data/synthetic/model_outputs.json')
    flag_rate = compute_flag_rate(outputs)
    if flag_rate > FLAG_RATE_THRESHOLD:
        print(f"High flag rate: {flag_rate:.2f}")
    long_responses = get_long_responses(outputs)
    print(f"Long responses: {len(long_responses)}")
    sample = random.sample(outputs, min(SAMPLE_SIZE, len(outputs)))
    for item in sample:
        print(item['model'], item['response'][:50])

In [ ]:
source = Path('clean_analysis.py').read_text()

In [ ]:
check_equal(Path('clean_analysis.py').exists(), True, "clean_analysis.py was created")
check_contains(source, 'SEED', "script defines SEED constant")
check_contains(source, 'def load_outputs', "script has load_outputs function")
check_contains(source, '"""', "script has docstrings")
check_contains(source, '__name__', "script has __name__ guard")

## Step 3 — Build the Reproducibility Report

Now document the cleaned analysis with a `reproducibility_report` dictionary. This is the artifact you would commit alongside the script so a future collaborator (or future you) knows exactly what was done and how to repeat it.

In [ ]:
reproducibility_report = {
    'seeds_set': True,
    'script_path': 'clean_analysis.py',
    'data_path': '../../data/synthetic/model_outputs.json',
    'constants_named': True,
    'has_docstrings': True
}

In [ ]:
check_keys(reproducibility_report, ['seeds_set', 'script_path', 'data_path', 'constants_named', 'has_docstrings'], "report has all required keys")
check_equal(reproducibility_report['seeds_set'], True, "seeds_set is True")
check_equal(Path(reproducibility_report['script_path']).exists(), True, "script_path points to a real file")

## Reflection — Module 14 Complete

Congratulations on completing Module 14. Here is a summary of the research coding habits you now have:

| Habit | Notebook |
|---|---|
| Know when to use a notebook vs a script | 01 |
| Extract reusable functions into `.py` modules | 01 |
| Use `snake_case` and `UPPER_SNAKE_CASE` | 02 |
| Write clear docstrings (purpose, args, returns) | 02 |
| Replace magic numbers with named constants | 02 |
| Apply the 5-item reproducibility checklist | 03 |
| Write a project README | 03 |
| Refactor messy code systematically | 04 |

These habits are not just good style — they are the foundation of trustworthy research. At safety labs, code that can't be understood and reproduced can't be relied on.

**Next up — Module 15: Research Memos**, where you will practise writing clear written summaries of quantitative findings for non-technical audiences.